# 05 · Bronze: daily snapshot of open ads

Takes a daily snapshot of all **currently open** ads for the project's search terms from the **JobTech JobSearch API**, and stores the raw responses in `lh_bronze`.

- **Input:** `https://jobsearch.api.jobtechdev.se/search`, same search terms as the historical backfill
- **Output:** `Files/bronze/jobsearch/snapshot_date=<YYYY-MM-DD>/term=<term>/ads.json`, same file format as `nb_01` (`_meta` + `hits`)
- **How to run:** *Run all*, or scheduled once a day from a Fabric pipeline. Re-running on the same day replaces that day's snapshot.
- **Lakehouse:** `lh_bronze` attached as default.

## Design notes

- **Why not the Historical Ads API for daily loads?** It lags: on 2026-11-03 it held 0–2 "data engineer" ads per day for the previous two weeks, while JobSearch held 2–8. It is filled in afterwards, so it is used for completed months only.
- **Why not JobStream?** JobStream has no free-text search; it returns all of Platsbanken or filters on occupation codes, and occupation codes do not identify the data roles reliably (see `nb_02`). Selecting ads differently from the backfill would put a break in the time series exactly where the daily pipeline starts. JobSearch accepts the same `q` search terms, so the selection stays the same.
- **Snapshots, not deltas.** Each run stores everything that is open right now (a few hundred ads), rather than only ads published since the last run. A missed day is caught up the next day as long as the ads are still open, and comparing consecutive snapshots shows when an ad was taken down, which gives how long ads actually stay open.
- **Same file format as the backfill**, so silver reads both sources with the same code.


## 1. Parameters

In [1]:
SEARCH_TERMS = [
    "data engineer", "data scientist", "dataanalytiker", "data analyst",
    "BI-utvecklare", "BI developer", '"analytics engineer"',
]

StatementMeta(, c0418d7d-ebe7-4a49-a344-5099ad3f810e, 3, Finished, Available, Finished, False)

## 2. Setup and helper functions

In [2]:
import json
import time
from datetime import datetime, timezone
from pathlib import Path
from zoneinfo import ZoneInfo

import requests

JOBSEARCH_URL = "https://jobsearch.api.jobtechdev.se/search"
PAGE_SIZE = 100
MAX_OFFSET = 2000
PAUSE_SECONDS = 0.3

SNAPSHOT_DATE = datetime.now(ZoneInfo("Europe/Stockholm")).date().isoformat()
BRONZE_REL = "Files/bronze/jobsearch"
LAKEHOUSE_ROOT = Path("/lakehouse/default")


def slug(term):
    """'"analytics engineer"' -> 'analytics_engineer' (safe folder name)."""
    return term.lower().replace('"', "").replace(" ", "_").replace("-", "_")


def rel_path(term):
    return f"{BRONZE_REL}/snapshot_date={SNAPSHOT_DATE}/term={slug(term)}/ads.json"


def fetch_page(params, retries=3):
    """One API call, retried with a short back-off on temporary errors."""
    for attempt in range(1, retries + 1):
        try:
            r = requests.get(JOBSEARCH_URL, params=params,
                             headers={"accept": "application/json"}, timeout=60)
            r.raise_for_status()
            return r.json()
        except requests.RequestException as e:
            print(f"  Attempt {attempt} failed: {e}")
            if attempt == retries:
                raise
            time.sleep(2 * attempt)


def fetch_open_ads(term):
    """All currently open ads for one search term, page by page."""
    hits, offset, total = [], 0, None
    while True:
        data = fetch_page({"q": term, "limit": PAGE_SIZE, "offset": offset})
        total = data["total"]["value"]
        page = data.get("hits", [])
        hits.extend(page)
        offset += PAGE_SIZE
        if not page or offset >= total:
            break
        if offset > MAX_OFFSET:
            print(f"  WARNING: {total} hits but the API only pages to ~{MAX_OFFSET}; saved {len(hits)}.")
            break
        time.sleep(PAUSE_SECONDS)
    return total, hits

print("Snapshot date:", SNAPSHOT_DATE)

StatementMeta(, c0418d7d-ebe7-4a49-a344-5099ad3f810e, 4, Finished, Available, Finished, False)

Snapshot date: 2026-11-03


## 3. Fetch and store today's snapshot

In [3]:
for term in SEARCH_TERMS:
    total, hits = fetch_open_ads(term)
    record = {
        "_meta": {
            "source": "jobtech_jobsearch_api",
            "search_term": term,
            "snapshot_date": SNAPSHOT_DATE,
            "fetched_at_utc": datetime.now(timezone.utc).isoformat(),
            "total_reported": total,
            "n_hits": len(hits),
        },
        "hits": hits,
    }
    out_file = LAKEHOUSE_ROOT / rel_path(term)
    out_file.parent.mkdir(parents=True, exist_ok=True)
    # Overwrite: re-running on the same day replaces that day's snapshot.
    out_file.write_text(json.dumps(record, ensure_ascii=False), encoding="utf-8")
    print(f"{term:22} {len(hits):4} of {total:4} open ads")

StatementMeta(, c0418d7d-ebe7-4a49-a344-5099ad3f810e, 5, Finished, Available, Finished, False)

data engineer            94 of   94 open ads
data scientist           41 of   41 open ads
dataanalytiker           30 of   30 open ads
data analyst             11 of   11 open ads
BI-utvecklare             9 of    9 open ads
BI developer             24 of   24 open ads
"analytics engineer"      7 of    7 open ads


## 4. Verify

In [4]:
problems, n_ads = [], 0
for term in SEARCH_TERMS:
    rp = rel_path(term)
    if not notebookutils.fs.exists(rp):
        problems.append((term, "missing file"))
        continue
    meta = json.loads((LAKEHOUSE_ROOT / rp).read_text(encoding="utf-8"))["_meta"]
    n_ads += meta["n_hits"]
    if meta["n_hits"] != meta["total_reported"]:
        problems.append((term, f"{meta['n_hits']} saved, {meta['total_reported']} reported"))

snapshots = sorted(f.name for f in notebookutils.fs.ls(BRONZE_REL))
print(f"Snapshot {SNAPSHOT_DATE}: {n_ads} ads (incl. overlap between terms)")
print(f"Snapshots stored so far: {len(snapshots)}  ({snapshots[0]} … {snapshots[-1]})")
print("Problems:", problems or "none")

assert not problems, "Daily snapshot verification failed"

StatementMeta(, c0418d7d-ebe7-4a49-a344-5099ad3f810e, 6, Finished, Available, Finished, False)

Snapshot 2026-11-03: 216 ads (incl. overlap between terms)
Snapshots stored so far: 1  (snapshot_date=2026-11-03 … snapshot_date=2026-11-03)
Problems: none
